In [1]:
import pandas as pd
import json
from pathlib import Path
from scipy.spatial.distance import jensenshannon
import numpy as np
from scipy.ndimage import zoom
import matplotlib.pyplot as plt


In [2]:
IMAGE_DIR = Path("/weka/eickhoff/esx139/flux_inpainting/flux_klein/consistent_set/qwen8/style0/cat_2/polarity0")

In [3]:
parquet_path = Path('/weka/eickhoff/esx139/flux_inpainting/flux_klein/consistent_set/qwen8/style0/cat_2/polarity0/ablation_results_style0_p0_with_swap.parquet')

In [4]:
df = pd.read_parquet(parquet_path)
df.head()

,sample_id,grid_h,grid_w,win_size,frac,stride,win_row,win_col,clean_probs,text-only_probs,correct_answer,variant,ablated-probs
0,03_29_1304_2_03,38,25,8,0.33,4,0,0,"{'A': 0.94921875, 'B': 0.047119140625, 'C': 0....","{'A': 4.6798959374427795e-08, 'B': 1.0, 'C': 5...",B,original_mean_ablation,"{'A': 0.7265625, 'B': 0.267578125, 'C': 0.0062..."
1,03_29_1304_2_03,38,25,8,0.33,4,0,0,"{'A': 0.94921875, 'B': 0.047119140625, 'C': 0....","{'A': 4.6798959374427795e-08, 'B': 1.0, 'C': 5...",B,swap_female_bg,"{'A': 0.96484375, 'B': 0.0291748046875, 'C': 0..."
2,03_29_1304_2_03,38,25,8,0.33,4,0,0,"{'A': 0.94921875, 'B': 0.047119140625, 'C': 0....","{'A': 4.6798959374427795e-08, 'B': 1.0, 'C': 5...",B,swap_male_bg,"{'A': 0.84765625, 'B': 0.1474609375, 'C': 0.00..."
3,03_29_1304_2_03,38,25,8,0.33,4,0,4,"{'A': 0.94921875, 'B': 0.047119140625, 'C': 0....","{'A': 4.6798959374427795e-08, 'B': 1.0, 'C': 5...",B,original_mean_ablation,"{'A': 0.93359375, 'B': 0.059814453125, 'C': 0...."
4,03_29_1304_2_03,38,25,8,0.33,4,0,4,"{'A': 0.94921875, 'B': 0.047119140625, 'C': 0....","{'A': 4.6798959374427795e-08, 'B': 1.0, 'C': 5...",B,swap_female_bg,"{'A': 0.96875, 'B': 0.0291748046875, 'C': 0.00..."


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9726 entries, 0 to 9725
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   sample_id        9726 non-null   object 
 1   grid_h           9726 non-null   int64  
 2   grid_w           9726 non-null   int64  
 3   win_size         9726 non-null   int64  
 4   frac             9726 non-null   float64
 5   stride           9726 non-null   int64  
 6   win_row          9726 non-null   int64  
 7   win_col          9726 non-null   int64  
 8   clean_probs      9726 non-null   object 
 9   text-only_probs  9726 non-null   object 
 10  correct_answer   9726 non-null   object 
 11  variant          9726 non-null   object 
 12  ablated-probs    9726 non-null   object 
dtypes: float64(1), int64(6), object(6)
memory usage: 987.9+ KB


In [ ]:
required_variants = {"original_mean_ablation", "swap_female_bg", "swap_male_bg"}

variant_col_candidates = ["variant", "variant_name", "ablation_variant", "edit_variant"]
variant_col = next((c for c in variant_col_candidates if c in df.columns), None)
if variant_col is None:
    raise KeyError(f"Could not find variant column. Tried: {variant_col_candidates}")

group_keys = ["sample_id", "win_row", "win_col"]
missing_group_keys = [k for k in group_keys if k not in df.columns]
if missing_group_keys:
    raise KeyError(f"Missing required grouping columns: {missing_group_keys}")

clean_col = "clean_probs"
ablated_col = "ablated-probs"
for col in [clean_col, ablated_col]:
    if col not in df.columns:
        raise KeyError(f"Missing required probability column: {col}")

def _to_prob_dict(x):
    if isinstance(x, dict):
        out = {}
        for k, v in x.items():
            try:
                out[str(k)] = float(v)
            except (TypeError, ValueError):
                return None
        return out
    try:
        arr = np.asarray(x, dtype=float).ravel()
    except (TypeError, ValueError):
        return None
    if arr.size == 0 or not np.isfinite(arr).all():
        return None
    return {str(i): float(v) for i, v in enumerate(arr)}

def _normalize_prob_dict(d):
    vals = np.array(list(d.values()), dtype=float)
    vals = np.clip(vals, 0.0, None)
    s = vals.sum()
    if s <= 0:
        return None
    return {k: float(v / s) for k, v in zip(d.keys(), vals)}


def _jsd_divergence(clean_probs, ablated_probs):
    p_raw = _to_prob_dict(clean_probs)
    q_raw = _to_prob_dict(ablated_probs)
    if p_raw is None or q_raw is None:
        return np.nan

    keys = sorted(set(p_raw.keys()) | set(q_raw.keys()))
    p_aligned = {k: p_raw.get(k, 0.0) for k in keys}
    q_aligned = {k: q_raw.get(k, 0.0) for k in keys}

    p = _normalize_prob_dict(p_aligned)
    q = _normalize_prob_dict(q_aligned)
    if p is None or q is None:
        return np.nan

    p_vec = np.array([p[k] for k in keys], dtype=float)
    q_vec = np.array([q[k] for k in keys], dtype=float)

    # scipy.spatial.distance.jensenshannon returns sqrt(JSD), so square to get JSD.
    return float(jensenshannon(p_vec, q_vec) ** 2)

df_sel = df[df[variant_col].isin(required_variants)].copy()
df_sel["jsd_clean_vs_ablated"] = df_sel.apply(
    lambda r: _jsd_divergence(r[clean_col], r[ablated_col]),
    axis=1,
)

df_valid = df_sel.dropna(subset=["jsd_clean_vs_ablated"]).copy()

agg = (
    df_valid.groupby(group_keys + [variant_col], as_index=False)["jsd_clean_vs_ablated"]
    .mean()
)

pivot = agg.pivot_table(
    index=group_keys,
    columns=variant_col,
    values="jsd_clean_vs_ablated",
)

required_order = ["original_mean_ablation", "swap_female_bg", "swap_male_bg"]
for v in required_order:
    if v not in pivot.columns:
        pivot[v] = np.nan
pivot = pivot[required_order]

comparison_jsd = pivot.dropna(subset=required_order).reset_index()
comparison_jsd = comparison_jsd.rename(columns={
    "original_mean_ablation": "jsd__original_mean_ablation",
    "swap_female_bg": "jsd__swap_female_bg",
    "swap_male_bg": "jsd__swap_male_bg",
})

comparison_jsd["jsd__delta_female_minus_original"] = (
    comparison_jsd["jsd__swap_female_bg"] - comparison_jsd["jsd__original_mean_ablation"]
)
comparison_jsd["jsd__delta_male_minus_original"] = (
    comparison_jsd["jsd__swap_male_bg"] - comparison_jsd["jsd__original_mean_ablation"]
)
comparison_jsd["jsd__delta_male_minus_female"] = (
    comparison_jsd["jsd__swap_male_bg"] - comparison_jsd["jsd__swap_female_bg"]
)

print(f"Rows in selected variants: {len(df_sel):,}")
print(f"Rows with valid JSD (clean_probs vs ablated-probs): {len(df_valid):,}")
print(f"Complete sample_id+window groups (all 3 variants present): {len(comparison_jsd):,}")

comparison_jsd.head()

Rows in selected variants: 9,726
Rows with valid JSD (clean_probs vs ablated-probs): 9,726
Complete sample_id+window groups (all 3 variants present): 3,242


variant,sample_id,win_row,win_col,jsd__original_mean_ablation,jsd__swap_female_bg,jsd__swap_male_bg,jsd__delta_female_minus_original,jsd__delta_male_minus_original,jsd__delta_male_minus_female
0,03_01_0000_2_01,0,0,9.292379e-07,1.210333e-06,1.300387e-06,2.810946e-07,3.711488e-07,9.005422e-08
1,03_01_0000_2_01,0,5,1.511435e-07,7.100063e-07,9.724028e-07,5.588628e-07,8.212593e-07,2.623965e-07
2,03_01_0000_2_01,0,10,5.164994e-08,4.939753e-07,6.316703e-07,4.423253e-07,5.800204e-07,1.376951e-07
3,03_01_0000_2_01,0,15,3.335336e-07,1.240867e-07,1.159307e-07,-2.094470e-07,-2.176029e-07,-8.155989e-09
4,03_01_0000_2_01,0,20,3.005886e-07,1.129666e-07,5.642660e-08,-1.876220e-07,-2.441620e-07,-5.653997e-08


In [16]:
case_cols = [
    "jsd__original_mean_ablation",
    "jsd__swap_female_bg",
    "jsd__swap_male_bg",
]

missing_case_cols = [c for c in case_cols if c not in comparison_jsd.columns]
if missing_case_cols:
    raise KeyError(f"Missing case columns in comparison_jsd: {missing_case_cols}")

top_tables = []
for case_col in case_cols:
    idx = comparison_jsd.groupby("sample_id")[case_col].idxmax()
    top_case = comparison_jsd.loc[idx, ["sample_id", "win_row", "win_col", case_col]].copy()
    top_case = top_case.rename(columns={case_col: "top_jsd"})
    top_case["case"] = case_col
    top_tables.append(top_case)

top_jsd_per_sample = pd.concat(top_tables, ignore_index=True)
top_jsd_per_sample = top_jsd_per_sample[["case", "sample_id", "win_row", "win_col", "top_jsd"]]
top_jsd_per_sample = top_jsd_per_sample.sort_values(["case", "top_jsd"], ascending=[True, False]).reset_index(drop=True)

print(f"Rows (case x sample_id): {len(top_jsd_per_sample):,}")
print(f"Unique sample_id: {top_jsd_per_sample['sample_id'].nunique():,}")

# Top rows across sample_id for each case.
top10_per_case = top_jsd_per_sample.groupby("case", group_keys=False).head(10)
top10_per_case

Rows (case x sample_id): 255
Unique sample_id: 85


variant,case,sample_id,win_row,win_col,top_jsd
0,jsd__original_mean_ablation,03_42_3908_2_01,0,25,0.689701
1,jsd__original_mean_ablation,03_37_2920_2_03,2,6,0.591936
2,jsd__original_mean_ablation,03_29_1444_2_02,8,8,0.526703
3,jsd__original_mean_ablation,03_29_1304_2_03,12,8,0.514721
4,jsd__original_mean_ablation,03_13_0284_2_01,3,6,0.490764
5,jsd__original_mean_ablation,03_20_0628_2_01,16,24,0.365699
6,jsd__original_mean_ablation,03_26_0820_2_01,0,18,0.305494
7,jsd__original_mean_ablation,03_26_0812_2_01,0,18,0.294608
8,jsd__original_mean_ablation,03_29_1340_2_02,3,12,0.145095
9,jsd__original_mean_ablation,03_28_1076_2_02,5,0,0.075638
